# arXiv RAG: данные, поиск, оценка качества

Исследовательский ноутбук проекта. Здесь показан путь от сырого CSV до метрик поиска; рабочий код вынесен в `rag.py`, `prepare_data.py`, `evaluate.py`.

**Окружение:** Google Colab с GPU (T4). Для LLM-части нужна Ollama (раздел 7). Файл `arxiv_data.csv` (колонки `titles`, `summaries`, `terms`) положите рядом с ноутбуком; в репозиторий он не входит.


In [ ]:
%pip install -q chromadb sentence-transformers openai pyarrow
DATA = "."   # папка с arxiv_data.csv и результатами

## 1. Данные

In [ ]:
import pandas as pd
import re
from pathlib import Path

arxiv_data = pd.read_csv('arxiv_data.csv')
arxiv_data.columns
arxiv_data.head()

,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


In [ ]:
arxiv_data.isnull().sum()

,0
titles,0
summaries,0
terms,0


## 2. Очистка




In [ ]:
import pandas as pd
import re
from pathlib import Path


class ArxivDataCleaner:
    """Очистка и подготовка сырых данных для RAG-пайплайна."""

    def __init__(self, raw_path: str, categories: list[str] | None = None):
        self.raw_path = Path(raw_path)
        self.categories = categories
        self.df: pd.DataFrame | None = None

    def load(self) -> "ArxivDataCleaner":
        """Загрузка сырых данных из CSV."""
        self.df = pd.read_csv(self.raw_path)
        self.df = self.df.rename(columns={
            "summaries": "abstract",
            "terms": "categories"
        })
        return self

    def filter_by_category(self) -> "ArxivDataCleaner":
        """Фильтрация по нужным категориям"""
        if self.categories:
            pattern = '|'.join(self.categories)
            self.df = self.df[self.df['categories'].str.contains(pattern, na=False)]
        return self

    def drop_duplicates(self) -> "ArxivDataCleaner":
        self.df = self.df.drop_duplicates()
        return self

    def clean_text_fields(self) -> "ArxivDataCleaner":
        self.df['abstract_clean'] = self.df['abstract'].apply(self._clean_text)
        self.df['title_clean'] = self.df['titles'].apply(self._clean_text)
        return self

    def filter_short_abstracts(self, min_len: int = 50) -> "ArxivDataCleaner":
        self.df = self.df[self.df['abstract_clean'].str.len() > min_len]
        return self

    @staticmethod
    def _clean_text(text: str) -> str:
        text = re.sub(r'\$.*?\$', '', text)
        text = re.sub(r'\\[a-zA-Z]+\{.*?\}', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def run_pipeline(self) -> pd.DataFrame:
        """Запускает все шаги очистки последовательно."""
        return (
            self.load()
                .filter_by_category()
                .drop_duplicates()
                .clean_text_fields()
                .filter_short_abstracts()
                .df
        )

    def save(self, output_path: str) -> None:
        if self.df is None:
            raise ValueError("Нет данных для сохранения — сначала вызовите run_pipeline()")
        self.df.to_parquet(output_path, index=False)

In [ ]:
cleaner = ArxivDataCleaner(raw_path="arxiv_data.csv", categories=["cs.CL", "cs.LG"])
arxiv_data_clean = cleaner.run_pipeline()
cleaner.save("arxiv_clean.parquet")
len(arxiv_data_clean)

## 3. Эмбеддинги и индекс Chroma

Модель `intfloat/multilingual-e5-base`. Для E5 обязательны префиксы: `passage: ` у документов и `query: ` у запросов, векторы нормализуются.
Если готового `embeddings.npy` нет или он не совпадает по длине с таблицей, векторы считаются заново (минуты на GPU).

In [ ]:
import numpy as np, pandas as pd, chromadb, os, torch
from sentence_transformers import SentenceTransformer
from openai import OpenAI

df = pd.read_parquet(f"{DATA}/arxiv_clean.parquet").reset_index(drop=True)
model = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda" if torch.cuda.is_available() else "cpu")

emb_path = f"{DATA}/embeddings.npy"
emb = np.load(emb_path) if os.path.exists(emb_path) else None
if emb is None or len(emb) != len(df):
    print("Считаю эмбеддинги заново (несколько минут на GPU)...")
    texts = ("passage: " + df["title_clean"] + ". " + df["abstract_clean"]).tolist()
    emb = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    np.save(emb_path, emb)
print("Векторы:", emb.shape, "| статей:", len(df))

client = chromadb.EphemeralClient()
col = client.get_or_create_collection("arxiv", metadata={"hnsw:space": "cosine"})
B = 500
for i in range(0, len(df), B):
    part = df.iloc[i:i+B]
    col.add(
        ids=[str(j) for j in part.index],
        embeddings=emb[i:i+B].tolist(),
        documents=part["abstract_clean"].tolist(),
        metadatas=[{"title": t, "categories": c}
                   for t, c in zip(part["title_clean"], part["categories"])],
    )
print("В базе:", col.count())

Векторы: (22521, 768) | статей: 22521
В базе: 22521


## 4. Функция поиска: запрос → эмбеддинг → top-k

In [ ]:
MIN_TOP1 = 0.82

def search(query, k=5):
    q = model.encode("query: " + query.strip(), normalize_embeddings=True)
    res = col.query(query_embeddings=[q.tolist()], n_results=k,
                    include=["documents", "metadatas", "distances"])
    return [{"title": m["title"], "categories": m["categories"],
             "score": round(1 - d, 3), "text": doc}
            for doc, m, d in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])]

def search_filtered(query, k=5):
    r = search(query, k)
    return r if r and r[0]["score"] >= MIN_TOP1 else []

In [ ]:
for r in search("semi-supervised image segmentation", k=3):
    print(r["score"], r["title"])
    print("   ", r["text"][:150], "...\n")

0.882 Semi-Supervised Semantic Image Segmentation with Self-correcting Networks
    Building a large image dataset with high-quality object masks for semantic segmentation is costly and time consuming. In this paper, we introduce a pr ...

0.874 Semi-Supervised Medical Image Segmentation via Learning Consistency under Transformations
    The scarcity of labeled data often limits the application of supervised deep learning techniques for medical image segmentation. This has motivated th ...

0.87 Medical Image Segmentation via Unsupervised Convolutional Neural Network
    For the majority of the learning-based segmentation methods, a large quantity of high-quality training data is required. In this paper, we present a n ...



## 5. Ручная оценка поиска: 14 запросов × топ-5

Запросы разных типов: прямые, перефразированные, русскоязычные, вне домена. Каждому результату ставится оценка: **2** статья по теме, **1** рядом, **0** не по теме.
Ячейки с выводом очищены: `run_tests` печатает 70 результатов, а разметка интерактивная.

In [ ]:
tests = [
    ("semi-supervised medical image segmentation", ["segmentation", "semi-supervised"]),
    ("object detection in autonomous driving", ["detection", "driving"]),
    ("generative adversarial networks for image synthesis", ["adversarial", "gan"]),
    ("self-supervised representation learning", ["self-supervised", "contrastive"]),
    ("federated learning privacy", ["federated", "privacy"]),

    ("how to train a model when only few images are labeled", ["semi-supervised", "few", "label"]),
    ("making neural networks smaller and faster", ["compression", "pruning", "quantization", "efficient"]),

    ("graph neural networks for node classification", ["graph", "node"]),
    ("reinforcement learning robot navigation", ["reinforcement", "navigation", "robot"]),
    ("adversarial attacks on deep networks", ["adversarial", "attack", "robust"]),

    ("сегментация медицинских изображений", ["segmentation", "medical"]),
    ("обучение с подкреплением", ["reinforcement"]),

    ("deep learning", []),
    ("recipe for chocolate cake", []),
]

In [ ]:
def run_tests(tests, k=5, preview=140):
    rows = []
    for qi, (query, keywords) in enumerate(tests, 1):
        print("=" * 90)
        print(f"[{qi}] {query}")
        results = search(query, k=k)
        hits = 0
        for rank, r in enumerate(results, 1):
            text = (r["title"] + " " + r["text"]).lower()
            kw_hit = any(w in text for w in keywords) if keywords else None
            hits += bool(kw_hit)
            mark = {True: "✓", False: "✗", None: "?"}[kw_hit]
            print(f"  {rank}. [{mark}] {r['score']:.3f}  {r['title'][:80]}")
            print(f"        {r['text'][:preview]}...")
            rows.append({"query": query, "rank": rank, "score": r["score"],
                         "title": r["title"], "kw_hit": kw_hit, "my_label": ""})
        if keywords:
            print(f"  --> совпадений по ключевым словам: {hits}/{k}")
    return pd.DataFrame(rows)

report = run_tests(tests)

In [ ]:
import pandas as pd

report = pd.read_csv("search_eval.csv") if "report" not in globals() else report
report["my_label"] = pd.to_numeric(report["my_label"], errors="coerce")

for i, row in report.iterrows():
    if pd.notna(row["my_label"]):
        continue  # уже оценено, пропускаем
    print("\n" + "-" * 80)
    print(f"ЗАПРОС: {row['query']}   (место {row['rank']}, score {row['score']})")
    print(f"СТАТЬЯ: {row['title']}")
    ans = input("Оценка 2 / 1 / 0 (Enter - пропустить, q - выйти): ").strip()
    if ans == "q":
        break
    if ans in ("0", "1", "2"):
        report.at[i, "my_label"] = int(ans)

report.to_csv("search_eval.csv", index=False)
print("Сохранено")

In [ ]:
import pandas as pd

ev = pd.read_csv("search_eval.csv")
ev["my_label"] = pd.to_numeric(ev["my_label"], errors="coerce")

print("Без оценки:", ev["my_label"].isna().sum())

print("\nДоля релевантных в топ-5:")
print(ev.groupby("query")["my_label"].apply(lambda s: (s == 2).mean()).round(2).to_string())
top3 = ev[ev["rank"] <= 3].groupby("query")["my_label"].apply(lambda s: (s == 2).any())
print(f"\nЗапросов с релевантным результатом в топ-3: {top3.sum()} из {len(top3)}")
print("Провальные запросы:", top3[~top3].index.tolist())

print("\nСредний score по оценке:")
print(ev.groupby("my_label")["score"].mean().round(3))

Без оценки: 0

Доля релевантных в топ-5:
query
adversarial attacks on deep networks                     0.6
deep learning                                            0.8
federated learning privacy                               0.2
generative adversarial networks for image synthesis      0.4
graph neural networks for node classification            0.6
how to train a model when only few images are labeled    0.2
making neural networks smaller and faster                0.0
object detection in autonomous driving                   0.4
recipe for chocolate cake                                0.2
reinforcement learning robot navigation                  0.6
self-supervised representation learning                  0.2
semi-supervised medical image segmentation               0.6
обучение с подкреплением                                 0.4
сегментация медицинских изображений                      0.8

Запросов с релевантным результатом в топ-3: 13 из 14
Провальные запросы: ['making neural networks 

**Что показала оценка**

- Запросов с релевантной статьёй в топ-3: 13 из 14 (запрос про торт к метрике по теме отношения не имеет, разметка по нему спорная).
- Средний cosine-score статей с оценками 0 / 1 / 2: 0,851 / 0,874 / 0,875. У E5 значения сжаты в диапазоне 0,80–0,90, поэтому порог по score почти не разделяет хорошее и плохое.
- Слабые темы (federated learning, self-supervised, few labels, сжатие сетей) не объясняются нехваткой данных: статей по ним сотни (проверка ниже).

In [ ]:
import pandas as pd

df = pd.read_parquet("arxiv_clean.parquet").reset_index(drop=True)
text_all = (df["title_clean"] + " " + df["abstract_clean"]).str.lower()
topics = {
    "federated learning":      r"federated",
    "self-supervised":         r"self-supervised|contrastive",
    "сжатие сетей":            r"compression|pruning|quantization|distillation|lightweight",
    "few labels / semi-sup.":  r"semi-supervised|few labels|limited label|weakly",
    "GAN":                     r"generative adversarial|\bgan\b|\bgans\b",
    "object detection":        r"object detection",
    "deep learning (для сравнения)": r"deep learning",
}

print(f"Всего статей в базе: {len(df)}\n")
for name, pattern in topics.items():
    n = text_all.str.contains(pattern, regex=True).sum()
    print(f"{name:32s} {n:6d}  ({n/len(df):.1%})")

ev = pd.read_csv("search_eval.csv")
print("\nScore по запросам (отсортировано по максимуму):")
print(ev.groupby("query")["score"].agg(["max", "mean", "min"]).round(3)
        .sort_values("max").to_string())

Всего статей в базе: 22521

federated learning                  197  (0.9%)
self-supervised                     757  (3.4%)
сжатие сетей                       1084  (4.8%)
few labels / semi-sup.              839  (3.7%)
GAN                                1772  (7.9%)
object detection                    806  (3.6%)
deep learning (для сравнения)      2938  (13.0%)

Score по запросам (отсортировано по максимуму):
                                                         max   mean    min
query                                                                     
recipe for chocolate cake                              0.797  0.790  0.786
обучение с подкреплением                               0.799  0.797  0.796
сегментация медицинских изображений                    0.862  0.849  0.845
making neural networks smaller and faster              0.869  0.867  0.865
how to train a model when only few images are labeled  0.875  0.870  0.867
semi-supervised medical image segmentation             0.886 

## 6. Reranking cross-encoder'ом

Векторы берут 20 кандидатов, `cross-encoder/ms-marco-MiniLM-L-6-v2` читает пару «запрос + статья» целиком и оставляет 5 лучших. Оценки reranker растянуты (около 3–10), в отличие от сжатых cosine-score E5.

In [ ]:
import torch
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")

def search_rerank(query, k_candidates=20, k_final=5):
    cands = search(query, k=k_candidates)
    pairs = [(query, c["title"] + ". " + c["text"]) for c in cands]
    scores = reranker.predict(pairs, batch_size=32)
    for c, s in zip(cands, scores):
        c["rerank"] = float(s)
    cands.sort(key=lambda c: c["rerank"], reverse=True)
    return cands[:k_final]

In [ ]:
weak = ["federated learning privacy",
        "self-supervised representation learning",
        "making neural networks smaller and faster",
        "how to train a model when only few images are labeled"]

for q in weak:
    print("\n" + "=" * 90 + f"\n{q}")
    print("--- было (только вектора) ---")
    for r in search(q, k=5):
        print(f"  {r['score']:.3f}  {r['title'][:85]}")
    print("--- стало (с reranking) ---")
    for r in search_rerank(q):
        print(f"  {r['rerank']:6.2f}  {r['title'][:85]}")


federated learning privacy
--- было (только вектора) ---
  0.888  Privacy Threats Analysis to Secure Federated Learning
  0.883  Federated Generative Privacy
  0.879  A Federated Learning Framework for Smart Grids: Securing Power Traces in Collaborativ
  0.876  Communication-Computation Efficient Secure Aggregation for Federated Learning
  0.874  Provable Defense against Privacy Leakage in Federated Learning from Representation Pe
--- стало (с reranking) ---
    9.04  Privacy Threats Analysis to Secure Federated Learning
    8.49  Federated Ensemble Model-based Reinforcement Learning
    7.96  Federated Generative Privacy
    7.77  Provable Defense against Privacy Leakage in Federated Learning from Representation Pe
    7.73  Large-Scale Secure XGB for Vertical Federated Learning

self-supervised representation learning
--- было (только вектора) ---
  0.897  Self-Supervised Representation Learning from Flow Equivariance
  0.894  Predicting What You Already Know Helps: Provable Self-Su

## 7. RAG-цепочка: перевод вопроса → поиск → ответ LLM

LLM запускается локально через Ollama (в Colab её нужно установить заново после каждого перезапуска сессии).
Вопрос переводится на английский, потому что абстракты английские.

In [ ]:
!apt-get install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(8)

!ollama pull qwen2.5:7b

In [ ]:
llm = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
LLM_MODEL = "qwen2.5:7b"

def build_prompt(question, results):
    context = "\n\n".join(f"[{i}] {r['title']}\n{r['text']}" for i, r in enumerate(results, 1))
    return f"""Ты помощник по научным статьям arXiv. Отвечай на вопрос ТОЛЬКО по статьям ниже.
Если среди них нет ответа, так и скажи: "В базе нет подходящих статей". Ничего не выдумывай.
Некоторые статьи могут быть не по теме, игнорируй их.
В конце перечисли номера и названия статей, которые использовал.

Статьи:
{context}

Вопрос: {question}
Отвечай строго на русском языке. Ответ:"""

def translate_to_en(text):
    r = llm.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "system", "content": "Translate the user's text to English. "
                   "Output only the translation. If it is already English, return it unchanged."},
                  {"role": "user", "content": text}])
    return r.choices[0].message.content.strip()

def answer(question, k=5):
    results = search_filtered(translate_to_en(question), k=k)
    if not results:
        return "В базе нет подходящих статей по этому вопросу."
    r = llm.chat.completions.create(
        model=LLM_MODEL, temperature=0.2,
        messages=[{"role": "user", "content": build_prompt(question, results)}])
    return r.choices[0].message.content.strip()

## 8. Тестовый набор с эталонными статьями

Идея: берём статью из базы, LLM придумывает вопрос на русском и эталонный ответ по аннотации, **вы проверяете вручную**. Правильным источником считается эта статья.




In [ ]:
import random, re

rng = random.Random(42)
cand = df[df["abstract_clean"].str.len().between(600, 1800)]
sample_idx = rng.sample(list(cand.index), 40)

def draft_question(row):
    prompt = f"""Ниже аннотация научной статьи. Придумай ОДИН вопрос на русском, который мог бы задать исследователь
и на который эта статья отвечает. Не используй слова из названия статьи и её редкие термины дословно, перефразируй.
Затем напиши эталонный ответ в 1-2 предложениях, строго по аннотации.

Формат строго такой:
ВОПРОС: ...
ОТВЕТ: ...

Название: {row['title_clean']}
Аннотация: {row['abstract_clean']}"""
    r = llm.chat.completions.create(model=LLM_MODEL, temperature=0.3,
                                    messages=[{"role": "user", "content": prompt}])
    m = re.search(r"ВОПРОС:\s*(.+?)\s*ОТВЕТ:\s*(.+)", r.choices[0].message.content, re.S)
    return (m.group(1).strip(), m.group(2).strip()) if m else None

In [ ]:
eval_set = []
for idx in sample_idx:
    if len(eval_set) >= 20:
        break
    row = df.loc[idx]
    d = draft_question(row)
    if d is None:
        continue
    q, ref = d
    print("\n" + "=" * 90)
    print(f"[{len(eval_set)+1}/20] СТАТЬЯ: {row['title_clean']}")
    print(f"АННОТАЦИЯ: {row['abstract_clean'][:450]}...")
    print(f"\nВОПРОС:  {q}\nЭТАЛОН:  {ref}")
    a = input("Enter = оставить, текст = своя формулировка вопроса, s = пропустить: ").strip()
    if a.lower() == "s":
        continue
    if a and len(a) < 15:      # защита от случайного ввода вроде слова 'текст'
        print('Слишком коротко для вопроса, пропускаю.')
        continue
    if a:
        q = a
    gold_ids = {str(i) for i in df.index[df["title_clean"] == row["title_clean"]]}
    eval_set.append({"question": q, "reference": ref, "gold_title": row["title_clean"],
                     "gold_ids": sorted(gold_ids)})

import json
json.dump(eval_set, open("eval_set.json", "w"), ensure_ascii=False, indent=2)
print("Сохранено вопросов:", len(eval_set))

## 9. Метрики поиска: только E5 и E5 + reranking

In [ ]:
import numpy as np

def retrieve(query_en, k=20):
    q = model.encode("query: " + query_en, normalize_embeddings=True)
    res = col.query(query_embeddings=[q.tolist()], n_results=k,
                    include=["documents", "metadatas"])
    return [{"id": i, "title": m["title"], "text": d}
            for i, m, d in zip(res["ids"][0], res["metadatas"][0], res["documents"][0])]

def rerank(query_en, cands):
    scores = reranker.predict([(query_en, c["title"] + ". " + c["text"]) for c in cands], batch_size=32)
    return [c for _, c in sorted(zip(scores, cands), key=lambda x: -x[0])]

def rank_of(gold_ids, ranked):
    for pos, c in enumerate(ranked, 1):
        if c["id"] in gold_ids:
            return pos
    return None

rows = []
for item in eval_set:
    q_en = translate_to_en(item["question"])
    base = retrieve(q_en, k=20)
    rr = rerank(q_en, base)
    gold = set(item["gold_ids"])
    rows.append({"question": item["question"], "q_en": q_en, "gold_title": item["gold_title"],
                 "rank_vec": rank_of(gold, base), "rank_rr": rank_of(gold, rr),
                 "top3_vec": [c["title"] for c in base[:3]],
                 "top3_rr": [c["title"] for c in rr[:3]]})
res = pd.DataFrame(rows)

def metrics(col_name):
    r = res[col_name]
    out = {f"Hit@{k}": round((r.notna() & (r <= k)).mean(), 2) for k in (1, 3, 5, 10)}
    out["MRR"] = round(np.where(r.notna(), 1 / r.fillna(1), 0).mean(), 3)
    out["Recall@20"] = round(r.notna().mean(), 2)
    return out

print(f"Вопросов: {len(res)}")
print("Только векторы (E5):", metrics("rank_vec"))
print("Векторы + reranking:", metrics("rank_rr"))
n3 = int((res["rank_rr"].notna() & (res["rank_rr"] <= 3)).sum())
print(f"\nС reranking: {n3} из {len(res)} вопросов нашли нужную статью в топ-3")

Вопросов: 20
Только векторы (E5): {'Hit@1': np.float64(0.25), 'Hit@3': np.float64(0.45), 'Hit@5': np.float64(0.55), 'Hit@10': np.float64(0.75), 'MRR': np.float64(0.396), 'Recall@20': np.float64(0.75)}
Векторы + reranking: {'Hit@1': np.float64(0.6), 'Hit@3': np.float64(0.65), 'Hit@5': np.float64(0.65), 'Hit@10': np.float64(0.75), 'MRR': np.float64(0.632), 'Recall@20': np.float64(0.75)}

С reranking: 13 из 20 вопросов нашли нужную статью в топ-3


**Коррекция.** Среди 20 вопросов два некорректных (их текст «текст», см. разбор промахов ниже). Они не нашли ничего ни в одном варианте поиска, поэтому дают нулевой вклад в каждую метрику и занижают результат. Ячейка ниже считает метрики только по корректным вопросам.

Пересчёт значений выше без этих двух вопросов (n = 18):

| Вариант | Hit@1 | Hit@3 | Hit@5 | Hit@10 | MRR | Recall@20 |
|---|---|---|---|---|---|---|
| Только E5 | 0,28 | 0,50 | 0,61 | 0,83 | 0,44 | 0,83 |
| E5 + reranking | 0,67 | 0,72 | 0,72 | 0,83 | 0,70 | 0,83 |

Reranking поднимает нужную статью в топ-1 и топ-3, но не помогает, если её нет среди 20 кандидатов (3 вопроса из 18): потолок задаёт первичный поиск.

In [ ]:
ok = res[res["question"].str.len() > 20].copy()   # отбрасываем некорректные вопросы вроде «текст»

def metrics_df(d, col_name):
    r = d[col_name]
    out = {f"Hit@{k}": round(float((r.notna() & (r <= k)).mean()), 2) for k in (1, 3, 5, 10)}
    out["MRR"] = round(float(np.where(r.notna(), 1 / r.fillna(1), 0).mean()), 3)
    out["Recall@20"] = round(float(r.notna().mean()), 2)
    return out

print(f"Корректных вопросов: {len(ok)} из {len(res)}")
print("Только E5:     ", metrics_df(ok, "rank_vec"))
print("E5 + reranking:", metrics_df(ok, "rank_rr"))

## 10. Разбор промахов

Из 7 промахов в топ-3: **2 некорректных вопроса** («текст»), **2 случая**, где нужная статья на местах 6–7 (в топ-3 при этом стоят близкие по теме статьи, поэтому Hit@k занижает качество), и **3 случая**, где нужной статьи нет даже среди 20 кандидатов (слишком общие вопросы или слабость эмбеддингов).

In [ ]:
miss = res[(res["rank_rr"].isna()) | (res["rank_rr"] > 3)]
for _, r in miss.iterrows():
    print("\nВОПРОС:", r["question"])
    print("EN:    ", r["q_en"])
    print("ЖДАЛИ: ", r["gold_title"], "| позиция:", r["rank_rr"])
    print("ПОЛУЧИЛИ:", *r["top3_rr"], sep="\n   - ")


ВОПРОС: текст
EN:     text
ЖДАЛИ:  Disentangled Representation Learning and Generation with Manifold Optimization | позиция: nan
ПОЛУЧИЛИ:
   - A survey on natural language processing (nlp) and applications in insurance
   - TextWorld: A Learning Environment for Text-based Games
   - Efficient Neural Architecture for Text-to-Image Synthesis

ВОПРОС: текст
EN:     text
ЖДАЛИ:  Two-Bit Networks for Deep Learning on Resource-Constrained Embedded Devices | позиция: nan
ПОЛУЧИЛИ:
   - A survey on natural language processing (nlp) and applications in insurance
   - TextWorld: A Learning Environment for Text-based Games
   - Efficient Neural Architecture for Text-to-Image Synthesis

ВОПРОС: Каким образом авторы повысили интерпретируемость и сжатие модели решений, не прибегая к обратному распространению в студенте?
EN:     How did the authors increase the interpretability and compression of the solution model without using backpropagation in the student?
ЖДАЛИ:  Rectified Decision Trees: Towa

## 11. Дальше

Бот и веб-интерфейс используют тот же код, вынесенный в `rag.py`: `python bot.py` (Telegram) и `streamlit run app.py`.